In [ ]:
import urllib.request
url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)


('the-verdict.txt', <http.client.HTTPMessage at 0x2205ff316f0>)

In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    text = f.read()
print(f"Length of text: {len(text)} characters")

Length of text: 20479 characters


## 正则化表达创建分词器

In [10]:
import re
preprocessed = re.split(r'([,.:;?!_"()\']|--|\s)', text)
preprocessed = [tok for tok in preprocessed if tok.strip()]
print(f"Number of tokens: {len(preprocessed)}")

Number of tokens: 4690


In [ ]:
all_words = sorted(preprocessed)
vocab_size = len(set(all_words))
vocab = {token: idx for idx, token in enumerate(set(all_words))}

724


## 分词器V1

In [15]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.vocab = vocab
        self.inv_vocab = {idx: token for token, idx in vocab.items()}

    def encode(self, text):
        tokens = re.split(r'([,.:;?!_"()\']|--|\s)', text)
        tokens = [tok for tok in tokens if tok.strip()]
        return [self.vocab[token] for token in tokens if token in self.vocab]

    def decode(self, token_ids):
        text = ' '.join([self.inv_vocab[token_id] for token_id in token_ids])
        text = re.sub(r'\s([,.:;?!_"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)
text = "The verdict is in."
ids = tokenizer.encode(text)

[847, 354, 721, 990]
The is in.


In [29]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(['<|endoftext|>', '<|unk|>'])
vocab = {token:i for i, token in enumerate(all_tokens)}

In [30]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(f"{i}: {item}")

0: ('younger', 1127)
1: ('your', 1128)
2: ('yourself', 1129)
3: ('<|endoftext|>', 1130)
4: ('<|unk|>', 1131)


In [31]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.vocab = vocab
        self.inv_vocab = {idx: token for token, idx in vocab.items()}

    def encode(self, text):
        tokens = re.split(r'([,.:;?!_"()\']|--|\s)', text)
        tokens = [tok for tok in tokens if tok.strip()]
        tokens = [item if item in self.vocab else '<|unk|>' for item in tokens]
        ids = [self.vocab[item] for item in tokens]
        return ids

    def decode(self, token_ids):
        text = ' '.join([self.inv_vocab[token_id] for token_id in token_ids])
        text = re.sub(r'\s([,.:;?!_"()\'])', r'\1', text)
        return text

In [34]:
tokenizer = SimpleTokenizerV2(vocab)
text = "The verditct is in."
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[93, 1131, 584, 568, 7]
The <|unk|> is in.


In [4]:
import tiktoken 
tokenizer = tiktoken.get_encoding("gpt2")

In [9]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces "
     "of someunknownPlace."
)
ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
string = tokenizer.decode(ids)
print(string)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.
